# Modelo inicial de calidad

Este experimento entrena un Random Forest con las imágenes de los tres archivos del conjunto público. La predicción se compara con la etiqueta de origen. `Mixed_source` significa que el recorte proviene de una foto de calidades mezcladas. Para evaluar una clase «Regular» hacen falta etiquetas por ejemplar.

Las imágenes se leen de las tres carpetas hermanas del repositorio. El experimento genera sus propias particiones y resultados a partir de los JPG. El clasificador usa color y textura. La evaluación del tamaño relativo requiere medidas o anotaciones independientes.

In [1]:
from datetime import datetime
import json
import math
from pathlib import Path
import re
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from PIL import Image, ImageOps
from scipy import ndimage
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.dataset_loader import (
    QUALITY_FOLDERS,
    build_dataset_inventory,
    extract_rgb_features,
    otsu_threshold,
    rgb_to_hsv_array,
)

DATA_ROOT = ROOT.parent
TARGET_SIZE = (128, 128)
LABELS = ["Bad", "Good", "Mixed_source"]
SEED = 42

inventory = build_dataset_inventory(DATA_ROOT)
if inventory.empty or set(inventory["quality"]) != set(QUALITY_FOLDERS.values()):
    raise FileNotFoundError(f"Se necesitan las tres carpetas de imágenes en {DATA_ROOT}")

print(f"JPG inventariados: {len(inventory):,}")
display(pd.crosstab(inventory["fruit"], inventory["quality"]))

JPG inventariados: 19,526


quality,Bad,Good,Mixed
fruit,,,
Apple,1141,1149,113
Banana,1087,1113,285
Guava,1129,1152,148
Lime,1085,1094,278
Orange,1159,1216,125
Pomegranate,1187,5940,125


## Recortes y descriptores

Para los recortes se usa el segmentador por saliencia HSV/Otsu y componentes conexas de la fase 3. Si el segmentador no encuentra un candidato, se omite la foto y se registra el conteo. Los candidatos pueden incluir regiones del fondo o frutas en contacto, por lo que la procedencia de la foto es una etiqueta débil.

Se reutiliza el extractor de morfometría, RGB, HSV y Laplaciano de `src/data`. Todas las entradas se redimensionan a 128 × 128 píxeles. Se excluyen las dimensiones de la imagen de entrada y la geometría del encuadre. Se usa color y textura para reducir el efecto del recorte y del formato de origen. En granada se conserva el archivo base si una variante con sufijo numérico de cuatro o más dígitos tiene un JPG base en la misma carpeta.

In [2]:
CAPTURE_STAMP = re.compile(r"(20\d{6})_?(\d{6})(?!\d)")
EXCLUDED_FEATURES = {
    "width", "height", "aspect_ratio", "area_ratio", "d_norm",
    "bbox_width_norm", "bbox_height_norm", "bbox_aspect_ratio",
}


def capture_timestamp(path, img):
    match = CAPTURE_STAMP.search(path.stem)
    if match:
        return pd.to_datetime("".join(match.groups()), format="%Y%m%d%H%M%S", errors="coerce")

    exif = img.getexif()
    value = exif.get_ifd(34665).get(36867) or exif.get(306)
    try:
        return datetime.strptime(value, "%Y:%m:%d %H:%M:%S")
    except (TypeError, ValueError):
        return pd.NaT


def derived_variant(path):
    stem, _, suffix = path.stem.rpartition("_")
    return (
        path.parent.name == "Pomegranate_Good"
        and suffix.isdecimal() and len(suffix) >= 4
        and path.with_stem(stem).is_file()
    )


def mixed_crops(img):
    rgb = np.asarray(img, dtype=np.float32) / 255.0
    h, w = rgb.shape[:2]
    border_px = max(3, min(h, w) // 20)
    border = np.concatenate([
        rgb[:border_px].reshape(-1, 3), rgb[-border_px:].reshape(-1, 3),
        rgb[:, :border_px].reshape(-1, 3), rgb[:, -border_px:].reshape(-1, 3),
    ])
    bg = np.median(border, axis=0)
    color_distance = np.linalg.norm(rgb - bg, axis=-1) / math.sqrt(3.0)
    saturation = rgb_to_hsv_array(rgb)[..., 1]
    salience = 0.55 * color_distance + 0.45 * saturation
    mask = salience > max(otsu_threshold(salience), 0.18)
    mask = ndimage.binary_fill_holes(ndimage.binary_opening(mask, structure=np.ones((5, 5))))
    components, _ = ndimage.label(mask)
    count = 0
    for label, region in enumerate(ndimage.find_objects(components), start=1):
        y, x = region
        height, width = y.stop - y.start, x.stop - x.start
        fraction = np.count_nonzero(components[region] == label) / (h * w)
        if not 0.018 <= fraction <= 0.65 or min(height, width) < 24:
            continue
        pad_y, pad_x = int(0.12 * height), int(0.12 * width)
        yield img.crop((
            max(0, x.start - pad_x), max(0, y.start - pad_y),
            min(w, x.stop + pad_x), min(h, y.stop + pad_y),
        ))
        count += 1
        if count == 5:
            break


def descriptors(img):
    resized = img.resize(TARGET_SIZE, Image.Resampling.BILINEAR)
    values = extract_rgb_features(np.asarray(resized, dtype=np.float32) / 255.0)
    return {name: value for name, value in values.items() if name not in EXCLUDED_FEATURES}

In [3]:
rows, unreadable, mixed_without_crop = [], [], []
excluded_variants = 0
for row in tqdm(inventory.itertuples(index=False), total=len(inventory), desc="Procesando JPG"):
    path = Path(row.filepath)
    relative_path = path.relative_to(DATA_ROOT).as_posix()
    if derived_variant(path):
        excluded_variants += 1
        continue

    try:
        with Image.open(path) as img:
            timestamp = capture_timestamp(path, img)
            if row.quality != "Mixed":
                img.draft("RGB", (256, 256))
            rgb = ImageOps.exif_transpose(img).convert("RGB")
        crops = list(mixed_crops(rgb)) if row.quality == "Mixed" else [rgb]
        if not crops:
            mixed_without_crop.append(relative_path)
            continue

        for crop_id, crop in enumerate(crops):
            rows.append({
                "source": relative_path,
                "capture_time": timestamp,
                "crop_id": crop_id,
                "fruit": row.fruit,
                "label": "Mixed_source" if row.quality == "Mixed" else row.quality,
                **descriptors(crop),
            })
    except OSError:
        unreadable.append(relative_path)

data = pd.DataFrame(rows)
data["capture_time"] = pd.to_datetime(data["capture_time"])
feature_names = [name for name in data if name not in {"source", "capture_time", "crop_id", "fruit", "label"}]
if not np.isfinite(data[feature_names].to_numpy()).all():
    raise ValueError("Se encontraron descriptores no finitos")
print(f"Variantes omitidas: {excluded_variants:,} | JPG ilegibles: {len(unreadable):,}")
print(f"Fotos Mixed sin recorte: {len(mixed_without_crop):,} | Ejemplos: {len(data):,}")
display(pd.crosstab(data["fruit"], data["label"]))

Procesando JPG:   0%|          | 0/19526 [00:00<?, ?it/s]

Variantes omitidas: 4,752 | JPG ilegibles: 0
Fotos Mixed sin recorte: 58 | Ejemplos: 15,772


label,Bad,Good,Mixed_source
fruit,,,
Apple,1141,1149,295
Banana,1087,1113,485
Guava,1129,1152,258
Lime,1085,1094,480
Orange,1159,1216,278
Pomegranate,1187,1188,276


## Particiones por grupos de captura

Se mantienen juntas las tomas de una misma fruta separadas por hasta 30 segundos, incluso si provienen de carpetas diferentes. La fecha se toma del nombre o de EXIF. Las fotos sin fecha se mantienen juntas por carpeta. La foto original determina el grupo de todos sus recortes.

Se eligen grupos completos para aproximar el reparto 70/15/15 y la distribución de fruta y etiqueta. Los grupos se construyen con las fechas disponibles y sirven como aproximación a sesiones de captura. La identidad física de cada fruta queda sin verificar.

In [4]:
photos = data[["source", "fruit", "capture_time"]].drop_duplicates("source").copy()
dated = photos[photos["capture_time"].notna()].sort_values(["fruit", "capture_time"]).copy()
gaps = dated.groupby("fruit")["capture_time"].diff().dt.total_seconds()
dated["group"] = dated["fruit"] + "/" + (gaps.isna() | gaps.gt(30)).cumsum().astype(str)
fallback = photos[photos["capture_time"].isna()].copy()
fallback["group"] = fallback["source"].map(lambda name: "undated/" + name.rsplit("/", 1)[0])
group_for_source = pd.concat([dated, fallback]).set_index("source")["group"]
data["group"] = data["source"].map(group_for_source)

def select_holdout(frame, fraction, seed):
    counts = pd.crosstab(frame["group"], [frame["label"], frame["fruit"]])
    targets = counts.sum(axis=0) * fraction
    quality_counts = counts.T.groupby(level="label").sum().T
    candidate_splits = GroupShuffleSplit(n_splits=400, test_size=fraction, random_state=seed)
    best = None
    for keep, holdout in candidate_splits.split(counts, groups=counts.index):
        if (quality_counts.iloc[holdout].sum() < 20).any() or (quality_counts.iloc[keep].sum() < 20).any():
            continue
        actual = counts.iloc[holdout].sum()
        penalty = float(((actual - targets).abs() / targets.clip(lower=1)).mean())
        penalty += abs(float(actual.sum() / counts.to_numpy().sum()) - fraction)
        if best is None or penalty < best[0]:
            best = (penalty, counts.index[holdout])
    if best is None:
        raise ValueError("No se pudo reservar una partición con las tres etiquetas")
    return best[1]


test_groups = select_holdout(data, 0.15, SEED)
development = data[~data["group"].isin(test_groups)]
val_groups = select_holdout(development, 0.15 / 0.85, SEED + 1)
data["split"] = np.select(
    [data["group"].isin(test_groups), data["group"].isin(val_groups)],
    ["test", "val"], default="train",
)

assert data.groupby("source")["split"].nunique().max() == 1
assert data.groupby("group")["split"].nunique().max() == 1
assert data.groupby("split")["label"].nunique().eq(len(LABELS)).all()

split_counts = pd.crosstab(data["split"], data["label"]).reindex(["train", "val", "test"])
display(split_counts.assign(total=split_counts.sum(axis=1)))
print(f"Grupos: {data['group'].nunique()} | Fotos sin fecha: {len(fallback):,}")

label,Bad,Good,Mixed_source,total
split,,,,
train,5059,4996,1908,11963
val,910,1174,73,2157
test,819,742,91,1652


Grupos: 210 | Fotos sin fecha: 793


## Entrenamiento y prueba

Random Forest utiliza pesos balanceados por clase. Se comparan tres configuraciones con F1 macro en validación y se usa la partición de prueba para medir el modelo elegido. La comparación de referencia predice la etiqueta más frecuente. Las métricas de prueba miden concordancia con las etiquetas del conjunto público.

In [5]:
train, val, test = [data[data["split"].eq(split)] for split in ("train", "val", "test")]
x_train, y_train = train[feature_names], train["label"]
x_val, y_val = val[feature_names], val["label"]
x_test, y_test = test[feature_names], test["label"]

trials = []
best_model = None
best_score = -1.0
for depth, min_leaf in [(10, 6), (16, 4), (24, 2)]:
    candidate = RandomForestClassifier(
        n_estimators=160, max_depth=depth, min_samples_leaf=min_leaf,
        class_weight="balanced_subsample", n_jobs=6, random_state=SEED,
    )
    candidate.fit(x_train, y_train)
    score = f1_score(y_val, candidate.predict(x_val), labels=LABELS, average="macro")
    trials.append({"max_depth": depth, "min_samples_leaf": min_leaf, "val_f1_macro": score})
    if score > best_score:
        best_model, best_score = candidate, score

display(pd.DataFrame(trials))
dummy = DummyClassifier(strategy="most_frequent").fit(x_train, y_train)
predictions = best_model.predict(x_test)
accuracy = accuracy_score(y_test, predictions)
test_f1 = f1_score(y_test, predictions, labels=LABELS, average="macro")
dummy_f1 = f1_score(y_test, dummy.predict(x_test), labels=LABELS, average="macro")
report = classification_report(y_test, predictions, labels=LABELS, zero_division=0, output_dict=True)
matrix = confusion_matrix(y_test, predictions, labels=LABELS)

print(f"F1 macro prueba: {test_f1:.4f} | Referencia mayoritaria: {dummy_f1:.4f} | Exactitud: {accuracy:.4f}")
display(pd.DataFrame(report).T.round(4))
display(pd.DataFrame(matrix, index=LABELS, columns=LABELS))

,max_depth,min_samples_leaf,val_f1_macro
0,10,6,0.611293
1,16,4,0.613469
2,24,2,0.612395


F1 macro prueba: 0.6616 | Referencia mayoritaria: 0.2210 | Exactitud: 0.7203


,precision,recall,f1-score,support
Bad,0.8578,0.6557,0.7433,819.0000
Good,0.6829,0.7925,0.7336,742.0000
Mixed_source,0.3939,0.7143,0.5078,91.0000
accuracy,0.7203,0.7203,0.7203,0.7203
macro avg,0.6449,0.7208,0.6616,1652.0000
weighted avg,0.7537,0.7203,0.7260,1652.0000


,Bad,Good,Mixed_source
Bad,537,248,34
Good,88,588,66
Mixed_source,1,25,65


In [6]:
checkpoint_path = ROOT / "experiments" / "checkpoints" / "quality_source_baseline.joblib"
metrics_path = ROOT / "experiments" / "results" / "quality_source_baseline_metrics.json"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.parent.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model": best_model,
    "feature_names": feature_names,
    "target_size": TARGET_SIZE,
    "labels": LABELS,
    "mixed_label_note": "Etiqueta de origen débil de fotos con calidades mezcladas",
    "sklearn_version": sklearn.__version__,
}
joblib.dump(checkpoint, checkpoint_path, compress=3)
metrics = {
    "label_order": LABELS,
    "mixed_label_note": checkpoint["mixed_label_note"],
    "sklearn_version": sklearn.__version__,
    "source_inventory": inventory["quality"].value_counts().to_dict(),
    "seed": SEED,
    "excluded_variants": excluded_variants,
    "unreadable_images": unreadable,
    "mixed_photos_without_crops": len(mixed_without_crop),
    "mixed_photos_with_crops": int(data.loc[data["label"].eq("Mixed_source"), "source"].nunique()),
    "undated_photos": len(fallback),
    "capture_groups_per_split": data.groupby("split")["group"].nunique().to_dict(),
    "split_counts": split_counts.to_dict(orient="index"),
    "split_counts_by_fruit": data.groupby(["split", "fruit", "label"]).size().unstack(fill_value=0).reset_index().to_dict(orient="records"),
    "validation_trials": trials,
    "selected_hyperparameters": {name: best_model.get_params()[name] for name in ("n_estimators", "max_depth", "min_samples_leaf")},
    "test_f1_macro": test_f1,
    "test_majority_f1_macro": dummy_f1,
    "test_accuracy": accuracy,
    "test_classification_report": report,
    "test_confusion_matrix": matrix.tolist(),
}
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
pd.testing.assert_index_equal(pd.Index(joblib.load(checkpoint_path)["feature_names"]), x_test.columns)
np.testing.assert_array_equal(joblib.load(checkpoint_path)["model"].predict(x_test.head()), predictions[:5])
print(checkpoint_path.relative_to(ROOT))
print(metrics_path.relative_to(ROOT))

experiments\checkpoints\quality_source_baseline.joblib
experiments\results\quality_source_baseline_metrics.json
